In [1]:

import torch

checkpoint_path = "best_checkpoint_seed123.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    print("\nFirst 30 model parameter names:")
    for key in list(state_dict.keys())[:30]:
        print(key)

    if "epoch" in checkpoint:
        print("\nSaved epoch:", checkpoint["epoch"])
    if "best_val_loss" in checkpoint:
        print("Best validation loss:", checkpoint["best_val_loss"])


Checkpoint type: <class 'dict'>
Top-level keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'best_val_loss', 'patience_counter']

First 30 model parameter names:
chemberta.embeddings.word_embeddings.weight
chemberta.embeddings.token_type_embeddings.weight
chemberta.embeddings.LayerNorm.weight
chemberta.embeddings.LayerNorm.bias
chemberta.embeddings.position_embeddings.weight
chemberta.encoder.layer.0.attention.self.query.weight
chemberta.encoder.layer.0.attention.self.query.bias
chemberta.encoder.layer.0.attention.self.key.weight
chemberta.encoder.layer.0.attention.self.key.bias
chemberta.encoder.layer.0.attention.self.value.weight
chemberta.encoder.layer.0.attention.self.value.bias
chemberta.encoder.layer.0.attention.output.dense.weight
chemberta.encoder.layer.0.attention.output.dense.bias
chemberta.encoder.layer.0.attention.output.LayerNorm.weight
chemberta.encoder.layer.0.attention.output.LayerNorm.bias
chemberta.encoder.layer.0.intermediate.dense.weight
chemberta.encoder.

In [2]:

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os

from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

MODEL_NAME = "DeepChem/ChemBERTa-100M-MLM"
MAX_LENGTH = 512
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


class HERG_ChemBERTa(nn.Module):
    def __init__(self):
        super().__init__()

        self.chemberta = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.chemberta.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.chemberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden_states = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden_states.dtype)

        masked_hidden = hidden_states * mask
        summed = masked_hidden.sum(dim=1)
        counts = mask.sum(dim=1)
        pooled = summed / counts.clamp(min=1)

        logits = self.classifier(pooled)
        return logits


# Load best checkpoint
checkpoint_path = "best_checkpoint_seed123.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

state_dict = (
    checkpoint["model_state_dict"]
    if "model_state_dict" in checkpoint
    else checkpoint
)

model = HERG_ChemBERTa()
model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Best ChemBERTa checkpoint loaded successfully.")


Device: cuda


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-100M-MLM and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Best ChemBERTa checkpoint loaded successfully.


In [3]:

model.eval()

test_smiles = ["CCO", "CC(=O)O"]

inputs = tokenizer(
    test_smiles,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    logits = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )
    probs = torch.sigmoid(logits.reshape(-1))

print("Logits:", logits.detach().cpu().numpy())
print("Probabilities:", probs.cpu().numpy())

assert len(probs) == 2
assert torch.isfinite(probs).all()
assert ((probs >= 0) & (probs <= 1)).all()

print("Sanity check passed.")


Logits: [[-2.2833703]
 [-2.3514497]]
Probabilities: [0.09250963 0.08695061]
Sanity check passed.


In [4]:

class SmilesDataset(Dataset):
    def __init__(self, smiles):
        self.smiles = list(smiles)

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        return self.smiles[idx]


def predict_chemberta_probabilities(smiles, batch_size=32):
    loader = DataLoader(
        SmilesDataset(smiles),
        batch_size=batch_size,
        shuffle=False
    )

    probabilities = []

    model.eval()
    with torch.no_grad():
        for batch_smiles in tqdm(loader):
            inputs = tokenizer(
                list(batch_smiles),
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            )

            inputs = {
                key: value.to(device)
                for key, value in inputs.items()
            }

            logits = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )

            probs = torch.sigmoid(logits.reshape(-1))
            probabilities.extend(probs.cpu().numpy())

    return np.asarray(probabilities)


val_df = pd.read_csv(
    "val.csv"
)
test_df = pd.read_csv(
    "test.csv"
)

chemberta_val_prob = predict_chemberta_probabilities(
    val_df["smiles"].tolist(),
    BATCH_SIZE
)

chemberta_test_prob = predict_chemberta_probabilities(
    test_df["smiles"].tolist(),
    BATCH_SIZE
)

print("Validation probabilities:", chemberta_val_prob.shape)
print("Test probabilities:", chemberta_test_prob.shape)

assert len(chemberta_val_prob) == len(val_df)
assert len(chemberta_test_prob) == len(test_df)


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/61 [00:00<?, ?it/s]

Validation probabilities: (954,)
Test probabilities: (1944,)


In [5]:

import os
import pandas as pd

# os.makedirs("herg_results", exist_ok=True)

pd.DataFrame({
    "smiles": val_df["smiles"],
    "y_true": val_df["hERG_blocker"].astype(int),
    "chemberta_probability": chemberta_val_prob
}).to_csv(
    "herg_results/chemberta_val_predictions.csv",
    index=False
)

pd.DataFrame({
    "smiles": test_df["smiles"],
    "y_true": test_df["hERG_blocker"].astype(int),
    "chemberta_probability": chemberta_test_prob
}).to_csv(
    "herg_results/chemberta_test_predictions.csv",
    index=False
)

print("ChemBERTa validation and test prediction files saved.")
print("Validation:", len(chemberta_val_prob))
print("Test:", len(chemberta_test_prob))


ChemBERTa validation and test prediction files saved.
Validation: 954
Test: 1944


In [6]:

import numpy as np
import pandas as pd
import os

# -----------------------------------------
# A. Load original validation/test datasets
# -----------------------------------------
val_df = pd.read_csv(
    "val.csv"
)
test_df = pd.read_csv(
    "test.csv"
)

y_val = val_df["hERG_blocker"].astype(int).to_numpy()
y_test = test_df["hERG_blocker"].astype(int).to_numpy()

# -----------------------------------------
# B. Load Chemprop probability predictions
# -----------------------------------------
chemprop_val_df = pd.read_csv(
    "val_predictions.csv"
)
chemprop_test_df = pd.read_csv(
    "test_predictions.csv"
)

# -----------------------------------------
# C. Load tuned XGBoost predictions
# -----------------------------------------
xgb_val_df = pd.read_csv(
    "xgboost_tuned_val_predictions.csv"
)
xgb_test_df = pd.read_csv(
    "xgboost_tuned_test_predictions.csv"
)

# -----------------------------------------
# D. Verify all prediction rows align
# -----------------------------------------
for name, pred_df, source_df in [
    ("Chemprop validation", chemprop_val_df, val_df),
    ("Chemprop test", chemprop_test_df, test_df),
    ("XGBoost validation", xgb_val_df, val_df),
    ("XGBoost test", xgb_test_df, test_df)
]:
    assert len(pred_df) == len(source_df), (
        f"{name}: row count mismatch"
    )
    assert pred_df["smiles"].equals(source_df["smiles"]), (
        f"{name}: SMILES/order mismatch"
    )

# Chemprop's target column contains predicted probabilities
chemprop_val_prob = chemprop_val_df["hERG_blocker"].to_numpy()
chemprop_test_prob = chemprop_test_df["hERG_blocker"].to_numpy()

xgb_val_prob_tuned = xgb_val_df["xgboost_probability"].to_numpy()
xgb_test_prob_tuned = xgb_test_df["xgboost_probability"].to_numpy()

# ChemBERTa arrays must already be available from your notebook.
# Expected variables:
# chemberta_val_prob
# chemberta_test_prob

assert len(chemberta_val_prob) == len(y_val)
assert len(chemberta_test_prob) == len(y_test)

# Check all probabilities are valid
for name, arr in [
    ("ChemBERTa validation", chemberta_val_prob),
    ("ChemBERTa test", chemberta_test_prob),
    ("Chemprop validation", chemprop_val_prob),
    ("Chemprop test", chemprop_test_prob),
    ("XGBoost validation", xgb_val_prob_tuned),
    ("XGBoost test", xgb_test_prob_tuned)
]:
    assert np.isfinite(arr).all(), f"{name}: non-finite values"
    assert ((arr >= 0) & (arr <= 1)).all(), (
        f"{name}: values outside [0, 1]"
    )

print("All prediction arrays are aligned and valid.")


All prediction arrays are aligned and valid.


In [7]:

# Validation ensemble probabilities
ensemble_val_prob = (
    chemberta_val_prob
    + chemprop_val_prob
    + xgb_val_prob_tuned
) / 3.0

# Test ensemble probabilities
ensemble_test_prob = (
    chemberta_test_prob
    + chemprop_test_prob
    + xgb_test_prob_tuned
) / 3.0

print("Validation probabilities:", ensemble_val_prob.shape)
print("Test probabilities:", ensemble_test_prob.shape)


Validation probabilities: (954,)
Test probabilities: (1944,)


In [8]:

from sklearn.metrics import confusion_matrix, matthews_corrcoef

def select_threshold(y_true, probabilities, target_sensitivity=0.90):
    rows = []

    # Include 0.0 so the all-positive prediction is considered
    thresholds = np.unique(
        np.concatenate(([0.0], probabilities))
    )

    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y_true, pred, labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn)
        specificity = tn / (tn + fp)
        mcc = matthews_corrcoef(y_true, pred)

        if sensitivity >= target_sensitivity:
            rows.append({
                "threshold": float(threshold),
                "sensitivity": sensitivity,
                "specificity": specificity,
                "MCC": mcc,
                "TN": tn, "FP": fp, "FN": fn, "TP": tp
            })

    if not rows:
        return None, pd.DataFrame()

    results = pd.DataFrame(rows).sort_values(
        ["MCC", "specificity"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return float(results.iloc[0]["threshold"]), results

ensemble_threshold, ensemble_threshold_results = select_threshold(
    y_val,
    ensemble_val_prob,
    target_sensitivity=0.90
)

if ensemble_threshold is None:
    print("No validation threshold achieved 90% sensitivity.")
else:
    print("Selected ensemble threshold:", ensemble_threshold)
    display(ensemble_threshold_results.head(10).round(4))


Selected ensemble threshold: 0.4334490458796438


,threshold,sensitivity,specificity,MCC,TN,FP,FN,TP
0,0.4334,0.9065,0.4639,0.4213,135,156,62,601
1,0.4347,0.9050,0.4639,0.4188,135,156,63,600
2,0.4322,0.9065,0.4605,0.4182,134,157,62,601
3,0.4316,0.9080,0.4570,0.4176,133,158,61,602
4,0.4355,0.9035,0.4639,0.4163,135,156,64,599
5,0.4322,0.9065,0.4570,0.4151,133,158,62,601
6,0.4315,0.9080,0.4536,0.4144,132,159,61,602
7,0.4356,0.9020,0.4639,0.4138,135,156,65,598
8,0.4310,0.9095,0.4502,0.4138,131,160,60,603
9,0.4302,0.9110,0.4467,0.4133,130,161,59,604


In [9]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_ensemble(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(
            y_true, pred, zero_division=0
        ),
        "Sensitivity": recall_score(
            y_true, pred, zero_division=0
        ),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities),
        "MCC": matthews_corrcoef(y_true, pred),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

if ensemble_threshold is not None:
    ensemble_test_results = evaluate_ensemble(
        y_test,
        ensemble_test_prob,
        ensemble_threshold
    )

    display(pd.DataFrame([ensemble_test_results]).round(4))

    os.makedirs("herg_results", exist_ok=True)

    pd.DataFrame({
        "smiles": test_df["smiles"],
        "y_true": y_test,
        "ensemble_probability": ensemble_test_prob
    }).to_csv(
        "ensemble_test_predictions.csv",
        index=False
    )

    pd.DataFrame({
        "smiles": val_df["smiles"],
        "y_true": y_val,
        "ensemble_probability": ensemble_val_prob
    }).to_csv(
        "ensemble_val_predictions.csv",
        index=False
    )

    print("Ensemble predictions saved.")


,Threshold,Accuracy,Precision,Sensitivity,Specificity,F1,ROC-AUC,PR-AUC,MCC,TN,FP,FN,TP
0,0.4334,0.7634,0.7766,0.918,0.4293,0.8414,0.7933,0.8887,0.4101,264,351,109,1220


Ensemble predictions saved.


#### this is for finding the weights to the models

In [10]:

import os
import numpy as np
import pandas as pd

from itertools import product
from sklearn.metrics import (
    confusion_matrix,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Change this to the folder containing your prediction CSVs.
RESULTS_DIR = "herg_results"

# -----------------------------------------
# 1. Load validation and test splits
# -----------------------------------------
# These should be the same scaffold splits used in training.
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

y_val = val_df["hERG_blocker"].astype(int).to_numpy()
y_test = test_df["hERG_blocker"].astype(int).to_numpy()


# -----------------------------------------
# 2. Load the probability prediction files
# -----------------------------------------
def load_predictions(filename, probability_column):
    return pd.read_csv(
        os.path.join(RESULTS_DIR, filename)
    )[["smiles", probability_column]]


cb_val = load_predictions(
    "chemberta_val_predictions.csv", "chemberta_probability"
)
cb_test = load_predictions(
    "chemberta_test_predictions.csv", "chemberta_probability"
)


# Set this to the directory containing your prediction CSVs
RESULTS_DIR = "."

# Chemprop predictions
cp_val = load_predictions(
    "val_predictions.csv", "hERG_blocker"
)
cp_test = load_predictions(
    "test_predictions.csv", "hERG_blocker"
)

# Tuned XGBoost predictions
xgb_val = load_predictions(
    "xgboost_tuned_val_predictions.csv",
    "xgboost_probability"
)
xgb_test = load_predictions(
    "xgboost_tuned_test_predictions.csv",
    "xgboost_probability"
)



# -----------------------------------------
# 3. Check row order and labels
# -----------------------------------------
for name, pred_df, source_df in [
    ("ChemBERTa validation", cb_val, val_df),
    ("ChemBERTa test", cb_test, test_df),
    ("Chemprop validation", cp_val, val_df),
    ("Chemprop test", cp_test, test_df),
    ("XGBoost validation", xgb_val, val_df),
    ("XGBoost test", xgb_test, test_df)
]:
    assert len(pred_df) == len(source_df), (
        f"{name}: row count mismatch"
    )
    assert pred_df["smiles"].reset_index(drop=True).equals(
        source_df["smiles"].reset_index(drop=True)
    ), f"{name}: SMILES order mismatch"

p_val = np.column_stack([
    cb_val["chemberta_probability"].to_numpy(),
    cp_val["hERG_blocker"].to_numpy(),
    xgb_val["xgboost_probability"].to_numpy()
])

p_test = np.column_stack([
    cb_test["chemberta_probability"].to_numpy(),
    cp_test["hERG_blocker"].to_numpy(),
    xgb_test["xgboost_probability"].to_numpy()
])

assert np.isfinite(p_val).all() and np.isfinite(p_test).all()
assert ((p_val >= 0) & (p_val <= 1)).all()
assert ((p_test >= 0) & (p_test <= 1)).all()

print("Prediction arrays aligned:", p_val.shape, p_test.shape)


# -----------------------------------------
# 4. Select a threshold for given weights
# -----------------------------------------
def best_threshold_for_weights(y_true, probabilities, target=0.90):
    thresholds = np.unique(
        np.concatenate(([0.0], probabilities))
    )

    best = None

    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y_true, pred, labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn)
        if sensitivity < target:
            continue

        specificity = tn / (tn + fp)
        mcc = matthews_corrcoef(y_true, pred)

        candidate = {
            "threshold": float(threshold),
            "sensitivity": sensitivity,
            "specificity": specificity,
            "MCC": mcc,
            "TN": tn, "FP": fp, "FN": fn, "TP": tp
        }

        if (
            best is None
            or candidate["MCC"] > best["MCC"]
            or (
                np.isclose(candidate["MCC"], best["MCC"])
                and candidate["specificity"] > best["specificity"]
            )
        ):
            best = candidate

    return best


# -----------------------------------------
# 5. Search weights in increments of 0.1
# -----------------------------------------
weight_values = np.arange(0, 1.01, 0.05)
weight_results = []

for w_cb in weight_values:
    for w_cp in weight_values:
        w_xgb = round(1.0 - w_cb - w_cp, 10)

        if w_xgb < -1e-9 or w_xgb > 1.0 + 1e-9:
            continue

        weights = np.array([w_cb, w_cp, max(0.0, w_xgb)])

        val_prob = p_val @ weights
        operating_point = best_threshold_for_weights(
            y_val, val_prob, target=0.90
        )

        if operating_point is None:
            continue

        weight_results.append({
            "w_ChemBERTa": weights[0],
            "w_Chemprop": weights[1],
            "w_XGBoost": weights[2],
            "val_PR_AUC": average_precision_score(y_val, val_prob),
            "val_ROC_AUC": roc_auc_score(y_val, val_prob),
            **operating_point
        })

weights_df = pd.DataFrame(weight_results).sort_values(
    ["MCC", "specificity"],
    ascending=[False, False]
).reset_index(drop=True)

display(weights_df.head(10).round(4))

if weights_df.empty:
    raise ValueError(
        "No weight combination achieved 90% validation sensitivity."
    )

best = weights_df.iloc[0]

best_weights = np.array([
    best["w_ChemBERTa"],
    best["w_Chemprop"],
    best["w_XGBoost"]
])

best_threshold = float(best["threshold"])

print("\nSelected weights:")
print("ChemBERTa:", best_weights[0])
print("Chemprop:", best_weights[1])
print("XGBoost:", best_weights[2])
print("Validation threshold:", best_threshold)
print("Validation sensitivity:", best["sensitivity"])
print("Validation MCC:", best["MCC"])


Prediction arrays aligned: (954, 3) (1944, 3)


,w_ChemBERTa,w_Chemprop,w_XGBoost,val_PR_AUC,val_ROC_AUC,threshold,sensitivity,specificity,MCC,TN,FP,FN,TP
0,0.10,0.00,0.90,0.9090,0.8171,0.5208,0.9035,0.5017,0.4503,146,145,64,599
1,0.20,0.00,0.80,0.9126,0.8203,0.4989,0.9050,0.4983,0.4497,145,146,63,600
2,0.00,0.10,0.90,0.9062,0.8160,0.5192,0.9155,0.4777,0.4489,139,152,56,607
3,0.05,0.00,0.95,0.9060,0.8141,0.5238,0.9020,0.5017,0.4478,146,145,65,598
4,0.15,0.00,0.85,0.9111,0.8193,0.5092,0.9020,0.4983,0.4448,145,146,65,598
5,0.05,0.10,0.85,0.9093,0.8189,0.5119,0.9140,0.4742,0.4433,138,153,57,606
6,0.00,0.15,0.85,0.9077,0.8179,0.5200,0.9065,0.4845,0.4399,141,150,62,601
7,0.00,0.05,0.95,0.9044,0.8134,0.5193,0.9110,0.4742,0.4382,138,153,59,604
8,0.00,0.00,1.00,0.9020,0.8099,0.5264,0.9035,0.4880,0.4380,142,149,64,599
9,0.05,0.05,0.90,0.9077,0.8168,0.5223,0.9035,0.4880,0.4380,142,149,64,599



Selected weights:
ChemBERTa: 0.1
Chemprop: 0.0
XGBoost: 0.9
Validation threshold: 0.520771132
Validation sensitivity: 0.9034690799396682
Validation MCC: 0.45025957867675853


In [11]:

# Combine test probabilities using validation-selected weights
ensemble_test_prob = p_test @ best_weights

test_pred = (ensemble_test_prob >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, test_pred, labels=[0, 1]
).ravel()

final_results = {
    "Threshold": best_threshold,
    "ChemBERTa weight": best_weights[0],
    "Chemprop weight": best_weights[1],
    "XGBoost weight": best_weights[2],
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(
        y_test, test_pred, zero_division=0
    ),
    "Sensitivity": recall_score(
        y_test, test_pred, zero_division=0
    ),
    "Specificity": tn / (tn + fp),
    "F1": f1_score(y_test, test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, ensemble_test_prob),
    "PR-AUC": average_precision_score(y_test, ensemble_test_prob),
    "MCC": matthews_corrcoef(y_test, test_pred),
    "TN": tn, "FP": fp, "FN": fn, "TP": tp
}

display(pd.DataFrame([final_results]).round(4))

# Save final artifacts
# os.makedirs(RESULTS_DIR, exist_ok=True)

weights_df.to_csv(
    os.path.join(RESULTS_DIR, "ensemble_weight_search.csv"),
    index=False
)

pd.DataFrame([final_results]).to_csv(
    os.path.join(RESULTS_DIR, "weighted_ensemble_test_metrics.csv"),
    index=False
)

pd.DataFrame({
    "smiles": test_df["smiles"],
    "y_true": y_test,
    "ensemble_probability": ensemble_test_prob
}).to_csv(
    os.path.join(RESULTS_DIR, "weighted_ensemble_test_predictions.csv"),
    index=False
)

pd.Series({
    "w_ChemBERTa": best_weights[0],
    "w_Chemprop": best_weights[1],
    "w_XGBoost": best_weights[2],
    "threshold": best_threshold
}).to_json(
    os.path.join(RESULTS_DIR, "weighted_ensemble_config.json"),
    indent=2
)

print("Final weighted ensemble artifacts saved.")


,Threshold,ChemBERTa weight,Chemprop weight,XGBoost weight,Accuracy,Precision,Sensitivity,Specificity,F1,ROC-AUC,PR-AUC,MCC,TN,FP,FN,TP
0,0.5208,0.1,0.0,0.9,0.7603,0.7882,0.8879,0.4846,0.8351,0.7738,0.8763,0.4116,298,317,149,1180


Final weighted ensemble artifacts saved.


In [12]:
merged_df = pd.read_csv('merged_dataset.csv')

merged_df

,Unnamed: 0,inchi_key,smiles,hERG_blocker
0,0,ZKLPARSLTMPFCP-UHFFFAOYSA-N,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,0
1,1,AAXGNXOJOSOUQQ-SANMLTNESA-N,CCCCN(Cc1cncn1Cc1ccc(C#N)cc1)[C@H]1CCN(Cc2cccc...,1
2,2,FPDIERBPQFAFSI-UHFFFAOYSA-N,CN1CCN(C2=Nc3ccccc3Nc3sc(CO)cc32)CC1,0
3,3,BRCCXAUQKZKZAB-QHCPKHFHSA-N,N#Cc1ccc(Cn2cncc2CN[C@H]2CCN(c3cccc4ncccc34)C2...,1
4,4,UIEATEWHFDRYRU-UHFFFAOYSA-N,CC(C)COCC(CN(Cc1ccccc1)c1ccccc1)N1CCCC1,1
...,...,...,...,...
10346,10346,ZRJRQVZRIPABBY-UHFFFAOYSA-N,N[C@@H](Cc1cccc(n1)C#N)c1ccccc1-c1noc2cc(ccc12...,1
10347,10347,ZRVYDDFEOLDULP-UHFFFAOYSA-N,FC1(F)CN(CC[C@@H]1CNc1ncccn1)C(=O)OC1Cc2ccccc2C1,0
10348,10348,ZSFYHUCRVPYFMX-UHFFFAOYSA-N,COc1cncc(OC[C@H]2CCNC2)c1-c1cc(Nc2cnc(cn2)C#N)...,0
10349,10349,ZSJAPINLVWEQHA-UHFFFAOYSA-N,N[C@@H](Cc1nc(ccc1F)-c1cn[nH]c1)c1ccccc1-c1noc...,1


In [28]:
first_df = pd.read_csv('herg_final_ML_dataset.csv')
first_df.head()
first_df.shape

(9806, 15)

In [29]:
binding_df = pd.read_csv('BindingDB_hERG_labeled_dataset.csv')
binding_df.head()
binding_df.shape

(1516, 20)

In [30]:
# print(first_df.columns)
first_df.shape

(9806, 15)

In [16]:
scaffold_val = pd.read_csv('val.csv')
scaffold_val

,Unnamed: 0,inchi_key,smiles,hERG_blocker
0,0,ZKLPARSLTMPFCP-UHFFFAOYSA-N,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,0
1,4,UIEATEWHFDRYRU-UHFFFAOYSA-N,CC(C)COCC(CN(Cc1ccccc1)c1ccccc1)N1CCCC1,1
2,22,VUFRFPUOFHXBBM-UHFFFAOYSA-N,Cc1ccc2c(-c3nnc(SCCCN4CC5CC5(c5ccc(Br)cc5)C4)n...,1
3,23,CLLQEXUCEWAACY-UHFFFAOYSA-N,Cc1ccc2c(-c3nnc(SCCCN4CC5CC5(c5ccc(C#N)cc5)C4)...,1
4,24,NKIUMOVWDJZMRB-UHFFFAOYSA-N,COc1ccc(C23CC2CN(CCCSc2nnc(-c4cccc5nc(C)ccc45)...,1
...,...,...,...,...
949,10285,WYASFUKPHAHHLE-UHFFFAOYSA-N,NCC1(O)CCN(C1)c1nc(Oc2cccc(F)c2)cc(n1)C(F)(F)F,1
950,10288,XADGEXPZZCPUIH-UHFFFAOYSA-N,CC(C)C(=O)N1CCc2c(C1)nc(C)n2C1CC2CCC(C1)N2CC[C...,1
951,10296,XHSSLADUNYJGQJ-UHFFFAOYSA-N,Cc1cc(cnc1N1Cc2ccc(OCC3COCCO3)cc2C1=O)C#Cc1ccccc1,1
952,10315,YDKYEYBUWOIAFU-UHFFFAOYSA-N,Nc1nnc([nH]1)N1CCC(CC1)N1C[C@H](Cn2ccnn2)OC[C@...,0


In [17]:
scaffold_train = pd.read_csv('train.csv')
scaffold_train

,Unnamed: 0,inchi_key,smiles,hERG_blocker
0,1,AAXGNXOJOSOUQQ-SANMLTNESA-N,CCCCN(Cc1cncn1Cc1ccc(C#N)cc1)[C@H]1CCN(Cc2cccc...,1
1,2,FPDIERBPQFAFSI-UHFFFAOYSA-N,CN1CCN(C2=Nc3ccccc3Nc3sc(CO)cc32)CC1,0
2,3,BRCCXAUQKZKZAB-QHCPKHFHSA-N,N#Cc1ccc(Cn2cncc2CN[C@H]2CCN(c3cccc4ncccc34)C2...,1
3,6,VNFPBHJOKIVQEB-UHFFFAOYSA-N,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,1
4,7,CCXVQWDBHGEOJC-NSHDSACASA-N,CC(C)CN(C(=O)c1cccc(C(F)(F)F)c1Cl)[C@H]1CCNC1,1
...,...,...,...,...
7446,10344,ZQPOIZTUINJPAQ-UHFFFAOYSA-N,Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)C(=O)N1CC(F)C1)C#C...,1
7447,10345,ZRDISNFFMHLYBW-UHFFFAOYSA-N,C[C@H]1C[C@H]2CSC(N)=N[C@]2(CO1)c1cc(NC(=O)c2c...,1
7448,10346,ZRJRQVZRIPABBY-UHFFFAOYSA-N,N[C@@H](Cc1cccc(n1)C#N)c1ccccc1-c1noc2cc(ccc12...,1
7449,10347,ZRVYDDFEOLDULP-UHFFFAOYSA-N,FC1(F)CN(CC[C@@H]1CNc1ncccn1)C(=O)OC1Cc2ccccc2C1,0


In [18]:
scaffold_train.columns

Index(['Unnamed: 0', 'inchi_key', 'smiles', 'hERG_blocker'], dtype='object')

In [37]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Load datasets
# ==========================================
first_df = pd.read_csv("herg_final_ML_dataset.csv")
binding_df = pd.read_csv("BindingDB_hERG_labeled_dataset.csv")

print("Original HERG shape:", first_df.shape)
print("Original BindingDB shape:", binding_df.shape)


# ==========================================
# 2. Select and standardize columns
# ==========================================
herg = first_df[["inchi_key", "smiles", "median_pIC50"]].copy()

binding = binding_df[
    ["Ligand InChI Key", "Ligand SMILES", "median_pIC50"]
].copy()

binding = binding.rename(columns={
    "Ligand InChI Key": "inchi_key",
    "Ligand SMILES": "smiles"
})


# ==========================================
# 3. Clean InChIKeys, SMILES and pIC50
# ==========================================
for df in [herg, binding]:
    df["inchi_key"] = df["inchi_key"].astype("string").str.strip()
    df["smiles"] = df["smiles"].astype("string").str.strip()
    df["median_pIC50"] = pd.to_numeric(
        df["median_pIC50"], errors="coerce"
    )

    df.loc[df["inchi_key"].eq(""), "inchi_key"] = pd.NA
    df.loc[df["smiles"].eq(""), "smiles"] = pd.NA

print("\nMissing values before dropping:")
print("HERG:\n", herg.isna().sum())
print("BindingDB:\n", binding.isna().sum())


# ==========================================
# 4. Remove rows without keys or SMILES
# ==========================================
herg = herg.dropna(subset=["inchi_key", "smiles"]).copy()
binding = binding.dropna(subset=["inchi_key", "smiles"]).copy()

print("\nRows after removing missing keys/SMILES:")
print("HERG:", len(herg))
print("BindingDB:", len(binding))


# ==========================================
# 5. Combine the datasets (UNION)
# ==========================================
combined = pd.concat([herg, binding], ignore_index=True)

print("\nCombined rows before deduplication:", len(combined))
print("Unique InChIKeys before deduplication:",
      combined["inchi_key"].nunique())


# ==========================================
# 6. Inspect duplicate InChIKeys
# ==========================================
duplicate_mask = combined["inchi_key"].duplicated(keep=False)

duplicate_records = combined.loc[duplicate_mask].sort_values("inchi_key")

print("\nRows belonging to duplicated InChIKeys:",
      len(duplicate_records))

print("Number of duplicated InChIKeys:",
      duplicate_records["inchi_key"].nunique())

display(duplicate_records.head(20))

Original HERG shape: (9806, 15)
Original BindingDB shape: (1516, 20)

Missing values before dropping:
HERG:
 inchi_key       0
smiles          0
median_pIC50    0
dtype: int64
BindingDB:
 inchi_key       197
smiles            0
median_pIC50      0
dtype: int64

Rows after removing missing keys/SMILES:
HERG: 9806
BindingDB: 1319

Combined rows before deduplication: 11125
Unique InChIKeys before deduplication: 10350

Rows belonging to duplicated InChIKeys: 1550
Number of duplicated InChIKeys: 775


,inchi_key,smiles,median_pIC50
5019,AACWUFIIMOHGSO-UHFFFAOYSA-N,Cc1cc(C)n(-c2cc(NC(=O)CCN(C)C)nc(-c3ccc(C)o3)n...,6.187087
9806,AACWUFIIMOHGSO-UHFFFAOYSA-N,CN(C)CCC(=O)Nc1cc(nc(n1)-c1ccc(C)o1)-n1nc(C)cc1C,6.187087
4597,ABDLOXDCMBBRCJ-DEOSSOPVSA-N,NCCCCN(Cc1nccc2c3ccccc3n(CC(N)=O)c12)[C@H]1CCC...,5.744727
9808,ABDLOXDCMBBRCJ-DEOSSOPVSA-N,NCCCCN(Cc1nccc2c3ccccc3n(CC(N)=O)c12)[C@H]1CCC...,5.744727
8651,ABOXKYVSSWHNON-UHFFFAOYSA-N,CC(C)(C)c1nccn1Cc1cn(-c2ccc(Cl)c(Cl)c2)nn1,6.279014
9809,ABOXKYVSSWHNON-UHFFFAOYSA-N,CC(C)(C)c1nccn1Cc1cn(nn1)-c1ccc(Cl)c(Cl)c1,6.279014
8659,ABPKHQVFYTVEEG-YDEUYYDZSA-N,C[C@H]1OC(=O)[C@]2(CCC(N)=O)CC(F)(F)[C@@H](C)[...,4.221849
9810,ABPKHQVFYTVEEG-YDEUYYDZSA-N,C[C@H]1OC(=O)[C@]2(CCC(N)=O)CC(F)(F)[C@@H](C)[...,4.221849
9108,ABXMAAVUSZPZAU-ISKFKSNPSA-N,CNC[C@H](c1ccc2ccccc2c1)[C@H](OCc1nc(C)no1)c1c...,4.823909
9812,ABXMAAVUSZPZAU-ISKFKSNPSA-N,CNC[C@@H]([C@H](OCc1nc(C)no1)c1ccccc1)c1ccc2cc...,4.823909


In [38]:
# ==========================================
# 7. Compare pIC50 values for duplicate keys
# ==========================================
potency_check = (
    combined.dropna(subset=["median_pIC50"])
    .groupby("inchi_key")
    .agg(
        n_potency_values=("median_pIC50", "count"),
        min_pIC50=("median_pIC50", "min"),
        max_pIC50=("median_pIC50", "max"),
        median_pIC50=("median_pIC50", "median")
    )
    .reset_index()
)

potency_check["pIC50_range"] = (
    potency_check["max_pIC50"] - potency_check["min_pIC50"]
)

conflicts = potency_check[
    potency_check["pIC50_range"] > 0.5
]

print("Compounds with pIC50 differences greater than 0.5:",
      len(conflicts))

display(conflicts.head(20))

Compounds with pIC50 differences greater than 0.5: 57


,inchi_key,n_potency_values,min_pIC50,max_pIC50,median_pIC50,pIC50_range
86,AENUUQTUDHOBSJ-UHFFFAOYSA-N,2,5.537426,6.175224,5.856325,0.637797
568,BHOUUDZNGWHFHC-UHFFFAOYSA-N,2,5.510764,6.829738,6.170251,1.318975
700,BQEMIVBQABRGAT-UHFFFAOYSA-N,2,5.944270,6.787812,6.366041,0.843543
717,BRIBVXPWUHZXBX-UHFFFAOYSA-N,2,5.225204,6.374688,5.799946,1.149483
1007,CIMZHFATSQYFET-UHFFFAOYSA-N,2,5.237607,6.554396,5.896002,1.316789
1018,CJDZRSVYGKAPJM-UHFFFAOYSA-N,2,5.066331,5.705534,5.385932,0.639203
1159,CRTYJRKVIQBRIW-UHFFFAOYSA-N,2,6.038274,7.183759,6.611017,1.145484
1257,CYBVKIGZLGUYOT-UHFFFAOYSA-N,2,5.400122,5.958607,5.679365,0.558485
1342,DDHFRTPLOOOEAP-UHFFFAOYSA-N,2,5.222403,6.067019,5.644711,0.844617
1346,DDNUGAKPXYMUKR-UHFFFAOYSA-N,2,5.376065,6.583359,5.979712,1.207295


In [39]:
# ==========================================
# 8. Build one regression record per compound
# ==========================================

# Remove rows without potency labels
labelled = combined.dropna(subset=["median_pIC50"]).copy()

# Summarize potency values by compound
potency_summary = (
    labelled.groupby("inchi_key")
    .agg(
        median_pIC50=("median_pIC50", "median"),
        min_pIC50=("median_pIC50", "min"),
        max_pIC50=("median_pIC50", "max"),
        n_labels=("median_pIC50", "count")
    )
    .reset_index()
)

potency_summary["pIC50_range"] = (
    potency_summary["max_pIC50"] - potency_summary["min_pIC50"]
)

# Keep keys with agreeing potency values
accepted_keys = potency_summary.loc[
    potency_summary["pIC50_range"] <= 0.5,
    "inchi_key"
]

# Select one SMILES per InChIKey
smiles_lookup = (
    combined.dropna(subset=["inchi_key", "smiles"])
    .drop_duplicates(subset=["inchi_key"])
    [["inchi_key", "smiles"]]
)

regression_df = (
    smiles_lookup
    .merge(
        potency_summary[["inchi_key", "median_pIC50", "pIC50_range"]],
        on="inchi_key",
        how="inner",
        validate="one_to_one"
    )
)

regression_df = regression_df[
    regression_df["inchi_key"].isin(accepted_keys)
].drop(columns=["pIC50_range"]).reset_index(drop=True)

print("Final regression dataset shape:", regression_df.shape)
print("Unique compounds:", regression_df["inchi_key"].nunique())
display(regression_df.head())

regression_df.to_csv(
    "herg_regression_dataset.csv",
    index=False
)

print("Saved: herg_regression_dataset.csv")

Final regression dataset shape: (10293, 3)
Unique compounds: 10293


,inchi_key,smiles,median_pIC50
0,ZKLPARSLTMPFCP-UHFFFAOYSA-N,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,4.500000
1,AAXGNXOJOSOUQQ-SANMLTNESA-N,CCCCN(Cc1cncn1Cc1ccc(C#N)cc1)[C@H]1CCN(Cc2cccc...,6.721246
2,FPDIERBPQFAFSI-UHFFFAOYSA-N,CN1CCN(C2=Nc3ccccc3Nc3sc(CO)cc32)CC1,4.940000
3,BRCCXAUQKZKZAB-QHCPKHFHSA-N,N#Cc1ccc(Cn2cncc2CN[C@H]2CCN(c3cccc4ncccc34)C2...,5.086186
4,UIEATEWHFDRYRU-UHFFFAOYSA-N,CC(C)COCC(CN(Cc1ccccc1)c1ccccc1)N1CCCC1,6.366532


Saved: herg_regression_dataset.csv


In [40]:
# ==========================================
# 1. Load existing classification splits
# ==========================================
train_cls = pd.read_csv("train.csv")
val_cls = pd.read_csv("val.csv")
test_cls = pd.read_csv("test.csv")

# Normalize keys consistently
for df in [train_cls, val_cls, test_cls, regression_df]:
    df["inchi_key"] = df["inchi_key"].astype("string").str.strip()

# Check the regression dataset has one row per key
assert regression_df["inchi_key"].is_unique

In [41]:
# ==========================================
# 2. Create regression splits
# ==========================================
def create_regression_split(classification_split, regression_data):
    # Preserve only compounds in this classification split
    matched = classification_split[["inchi_key"]].merge(
        regression_data[["inchi_key", "smiles", "median_pIC50"]],
        on="inchi_key",
        how="inner",
        validate="one_to_one"
    )

    return matched.reset_index(drop=True)


train_reg = create_regression_split(train_cls, regression_df)
val_reg = create_regression_split(val_cls, regression_df)
test_reg = create_regression_split(test_cls, regression_df)

print("Regression split sizes:")
print("Train:", train_reg.shape)
print("Validation:", val_reg.shape)
print("Test:", test_reg.shape)

print("\nTotal regression compounds:",
      len(train_reg) + len(val_reg) + len(test_reg))

Regression split sizes:
Train: (7402, 3)
Validation: (948, 3)
Test: (1941, 3)

Total regression compounds: 10291


In [42]:
train_reg.to_csv("train_regression.csv", index=False)
val_reg.to_csv("val_regression.csv", index=False)
test_reg.to_csv("test_regression.csv", index=False)

print("Saved train_regression.csv")
print("Saved val_regression.csv")
print("Saved test_regression.csv")

Saved train_regression.csv
Saved val_regression.csv
Saved test_regression.csv


In [43]:
merged = pd.read_csv('merged_dataset.csv')
merged.shape

(10351, 4)

In [44]:
train_reg = pd.read_csv('train_regression.csv')
train_reg

,inchi_key,smiles,median_pIC50
0,AAXGNXOJOSOUQQ-SANMLTNESA-N,CCCCN(Cc1cncn1Cc1ccc(C#N)cc1)[C@H]1CCN(Cc2cccc...,6.721246
1,FPDIERBPQFAFSI-UHFFFAOYSA-N,CN1CCN(C2=Nc3ccccc3Nc3sc(CO)cc32)CC1,4.940000
2,BRCCXAUQKZKZAB-QHCPKHFHSA-N,N#Cc1ccc(Cn2cncc2CN[C@H]2CCN(c3cccc4ncccc34)C2...,5.086186
3,VNFPBHJOKIVQEB-UHFFFAOYSA-N,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,5.520000
4,CCXVQWDBHGEOJC-NSHDSACASA-N,CC(C)CN(C(=O)c1cccc(C(F)(F)F)c1Cl)[C@H]1CCNC1,5.385103
...,...,...,...
7397,ZQPOIZTUINJPAQ-UHFFFAOYSA-N,Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)C(=O)N1CC(F)C1)C#C...,6.408935
7398,ZRDISNFFMHLYBW-UHFFFAOYSA-N,C[C@H]1C[C@H]2CSC(N)=N[C@]2(CO1)c1cc(NC(=O)c2c...,6.397940
7399,ZRJRQVZRIPABBY-UHFFFAOYSA-N,N[C@@H](Cc1cccc(n1)C#N)c1ccccc1-c1noc2cc(ccc12...,5.537602
7400,ZRVYDDFEOLDULP-UHFFFAOYSA-N,FC1(F)CN(CC[C@@H]1CNc1ncccn1)C(=O)OC1Cc2ccccc2...,4.455932


In [45]:
train = pd.read_csv('train.csv')
train

,Unnamed: 0,inchi_key,smiles,hERG_blocker
0,1,AAXGNXOJOSOUQQ-SANMLTNESA-N,CCCCN(Cc1cncn1Cc1ccc(C#N)cc1)[C@H]1CCN(Cc2cccc...,1
1,2,FPDIERBPQFAFSI-UHFFFAOYSA-N,CN1CCN(C2=Nc3ccccc3Nc3sc(CO)cc32)CC1,0
2,3,BRCCXAUQKZKZAB-QHCPKHFHSA-N,N#Cc1ccc(Cn2cncc2CN[C@H]2CCN(c3cccc4ncccc34)C2...,1
3,6,VNFPBHJOKIVQEB-UHFFFAOYSA-N,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,1
4,7,CCXVQWDBHGEOJC-NSHDSACASA-N,CC(C)CN(C(=O)c1cccc(C(F)(F)F)c1Cl)[C@H]1CCNC1,1
...,...,...,...,...
7446,10344,ZQPOIZTUINJPAQ-UHFFFAOYSA-N,Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)C(=O)N1CC(F)C1)C#C...,1
7447,10345,ZRDISNFFMHLYBW-UHFFFAOYSA-N,C[C@H]1C[C@H]2CSC(N)=N[C@]2(CO1)c1cc(NC(=O)c2c...,1
7448,10346,ZRJRQVZRIPABBY-UHFFFAOYSA-N,N[C@@H](Cc1cccc(n1)C#N)c1ccccc1-c1noc2cc(ccc12...,1
7449,10347,ZRVYDDFEOLDULP-UHFFFAOYSA-N,FC1(F)CN(CC[C@@H]1CNc1ncccn1)C(=O)OC1Cc2ccccc2C1,0


In [46]:
# Check that every regression compound belongs to its corresponding
# classification split, and that no compound is missing from another split.

for name, cls_df, reg_df in [
    ("Train", train_cls, train_reg),
    ("Validation", val_cls, val_reg),
    ("Test", test_cls, test_reg),
]:
    cls_keys = set(cls_df["inchi_key"].dropna())
    reg_keys = set(reg_df["inchi_key"].dropna())

    print(f"\n{name}")
    print("Classification compounds:", len(cls_keys))
    print("Regression compounds:", len(reg_keys))
    print("Regression compounds outside classification split:",
          len(reg_keys - cls_keys))
    print("Classification compounds without regression labels:",
          len(cls_keys - reg_keys))


Train
Classification compounds: 7451
Regression compounds: 7402
Regression compounds outside classification split: 0
Classification compounds without regression labels: 49

Validation
Classification compounds: 954
Regression compounds: 948
Regression compounds outside classification split: 0
Classification compounds without regression labels: 6

Test
Classification compounds: 1943
Regression compounds: 1941
Regression compounds outside classification split: 0
Classification compounds without regression labels: 2
